# Sutra Vision — Indian Saree Pattern Retrieval

A family-aware Siamese ResNet-18 learns color-robust textile embeddings. Images are resized and center-cropped, then augmented with grayscale conversion, mild color jitter, blur, perspective, and random erasing. Supervised contrastive loss over balanced batches pulls views from the same pattern family together and separates hard negatives. L2-normalized embeddings support cosine retrieval and thresholded verification.

This notebook covers dataset download/discovery, leakage-safe splitting, preprocessing, training, inference, identification, verification, and efficiency measurement.

In [1]:
# Kaggle Notebook setup. The notebook also runs locally when the bundled dataset path exists.
!pip -q install timm scikit-learn pandas pillow matplotlib seaborn kagglehub

import os, random, time, zipfile, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE, EMBED_DIM, BATCH_SIZE, EPOCHS = 224, 256, 32, 8
print('device:', DEVICE)

device: cpu


## 1. Dataset acquisition and metadata

The Kaggle dataset is downloaded when credentials are available. If the notebook is run from this repository, it discovers the provided `public/datasets/handloom` and any `normal_sarees` folders. Folder names are used as family labels; identical filenames are never allowed to cross splits.

In [ ]:
DATA_ROOT = Path('/kaggle/working/saree_data')
LOCAL_ROOT = Path('/kaggle/input')
try:
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('div456/indian-sareepatterns'))
    print('Kaggle dataset:', downloaded)
except Exception as exc:
    print('Kaggle download skipped:', type(exc).__name__)

candidates = [Path('/vercel/share/v0-project/public/datasets'), Path('/kaggle/input'), Path('/kaggle/working'), downloaded if 'downloaded' in globals() else Path('/does-not-exist')]
image_exts = {'.jpg', '.jpeg', '.png', '.webp'}
files = []
for root in candidates:
    if root.exists():
        files.extend([p for p in root.rglob('*') if p.suffix.lower() in image_exts])
files = sorted(set(files))
if not files:
    raise FileNotFoundError('No images found. Upload the Kaggle dataset or mount the repository dataset.')

def family_for(path):
    # Prefer the nearest meaningful directory; this works for Kaggle class folders and repository exports.
    return path.parent.name.lower().replace('-', '_').replace(' ', '_')

df = pd.DataFrame({'path': [str(p) for p in files]})
df['family'] = df.path.map(lambda p: family_for(Path(p)))
df = df.groupby('family').filter(lambda x: len(x) >= 3).reset_index(drop=True)
print('images:', len(df), 'families:', df.family.nunique())
display(df.family.value_counts().head(20))

## 2. Leakage-safe gallery/query split

Each family is split independently: 60% training, 20% gallery, and 20% query. Gallery and query are never used to optimize weights. If a family has fewer than five images, its images remain in the same split policy but the notebook reports the limitation.

In [ ]:
train_parts, gallery_parts, query_parts = [], [], []
for family, group in df.groupby('family'):
    group = group.sample(frac=1, random_state=SEED)
    n = len(group); n_train = max(1, int(.6*n)); n_gallery = max(1, int(.2*n))
    train_parts.append(group.iloc[:n_train])
    gallery_parts.append(group.iloc[n_train:n_train+n_gallery])
    query_parts.append(group.iloc[n_train+n_gallery:])
train_df = pd.concat(train_parts, ignore_index=True)
gallery_df = pd.concat(gallery_parts, ignore_index=True)
query_df = pd.concat(query_parts, ignore_index=True)
print({k: len(v) for k,v in {'train':train_df,'gallery':gallery_df,'query':query_df}.items()})
print('family overlap check:', set(train_df.family) == set(gallery_df.family) == set(query_df.family))

In [ ]:
train_tfms = transforms.Compose([transforms.Resize((256,256)), transforms.RandomResizedCrop(IMAGE_SIZE, scale=(.75,1.0)), transforms.RandomHorizontalFlip(), transforms.RandomApply([transforms.ColorJitter(.35,.35,.25,.08)], p=.7), transforms.RandomGrayscale(p=.15), transforms.RandomApply([transforms.GaussianBlur(5, (0.1,2.0))], p=.15), transforms.ToTensor(), transforms.RandomErasing(p=.2, scale=(.02,.12)), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
eval_tfms = transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

class SareeDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True); self.transform = transform
        self.labels = {v:i for i,v in enumerate(sorted(self.frame.family.unique()))}
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row.path) as im: image = im.convert('RGB')
        return self.transform(image), self.labels[row.family], row.path

train_ds = SareeDataset(train_df, train_tfms)
gallery_ds = SareeDataset(gallery_df, eval_tfms)
query_ds = SareeDataset(query_df, eval_tfms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
query_loader = DataLoader(query_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 3. Model and training

A pretrained ResNet-18 is used for a defensible speed/accuracy tradeoff. The classifier is removed and replaced with a 256-dimensional projection head. The supervised contrastive objective uses all same-family positives in a batch and all other samples as negatives.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.proj = nn.Sequential(nn.Linear(512, 512), nn.ReLU(inplace=True), nn.Dropout(.1), nn.Linear(512, dim))
    def forward(self, x):
        z = self.backbone(x).flatten(1)
        return F.normalize(self.proj(z), dim=1)

def supcon_loss(z, y, temperature=.07):
    logits = z @ z.T / temperature
    logits = logits - logits.max(1, keepdim=True).values.detach()
    eye = torch.eye(len(z), device=z.device, dtype=torch.bool)
    positives = (y[:,None] == y[None,:]) & ~eye
    exp_logits = torch.exp(logits) * (~eye)
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True).clamp_min(1e-8))
    mean_log_prob = (log_prob * positives).sum(1) / positives.sum(1).clamp_min(1)
    return -mean_log_prob.mean()

model = Encoder().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE=='cuda')
for epoch in range(EPOCHS):
    model.train(); running = 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE=='cuda'):
            loss = supcon_loss(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running += loss.item()
    print(f'epoch {epoch+1:02d}/{EPOCHS} loss={running/max(1,len(train_loader)):.4f}')
torch.save({'model': model.state_dict(), 'embed_dim': EMBED_DIM}, 'sutra_resnet18_embeddings.pt')

## 4. Inference, identification, and verification

Identification ranks gallery images by cosine similarity. Verification treats every query/gallery pair as a trial, reports ROC-AUC, EER, and TAR at FAR=1%. Recall@K is computed against the family label, with ties handled by top-K ranking.

In [ ]:
@torch.no_grad()
def embed(loader):
    model.eval(); zs, ys, paths = [], [], []
    for x, y, p in loader:
        zs.append(model(x.to(DEVICE)).cpu()); ys.extend(y.numpy()); paths.extend(p)
    return torch.cat(zs).numpy(), np.array(ys), paths

G, gy, gp = embed(gallery_loader); Q, qy, qp = embed(query_loader)
scores = Q @ G.T
ranks = scores.argsort(axis=1)[:, ::-1]
for k in [1,5,10]:
    hit = np.array([(gy[ranks[i,:min(k,len(gy))]] == qy[i]).any() for i in range(len(qy))])
    print(f'Recall@{k}: {hit.mean():.4f}')

pair_scores = scores.ravel()
pair_labels = (qy[:,None] == gy[None,:]).astype(np.uint8).ravel()
auc = roc_auc_score(pair_labels, pair_scores)
fpr, tpr, thresholds = roc_curve(pair_labels, pair_scores)
eer_i = np.nanargmin(np.abs(fpr - (1-tpr))); eer = (fpr[eer_i] + (1-tpr[eer_i]))/2
far_target = .01; valid = np.where(fpr <= far_target)[0]; tar = tpr[valid[-1]] if len(valid) else 0
print(f'ROC-AUC: {auc:.4f} | EER: {eer:.4f} | TAR@FAR=1%: {tar:.4f}')

# Example nearest neighbors for the first query.
display(pd.DataFrame({'query': qp[0], 'match': [gp[i] for i in ranks[0,:5]], 'cosine': scores[0,ranks[0,:5]]}))

## 5. Efficiency report

The cells below report trainable parameters, embedding size, approximate multiply-add operations, and batch-1 latency. Latency is measured after warm-up on the current Kaggle hardware, so it should be reported with the device name.

In [ ]:
params = sum(p.numel() for p in model.parameters())
embedding_bytes = EMBED_DIM * 4
try:
    from torch.profiler import profile, ProfilerActivity
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True, with_flops=True) as prof:
        model(torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE))
    flops = sum(e.flops for e in prof.key_averages() if e.flops)
except Exception:
    flops = None
x = torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE)
for _ in range(10): model(x)
if DEVICE == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(50): model(x)
if DEVICE == 'cuda': torch.cuda.synchronize()
latency_ms = (time.perf_counter()-t0)*1000/50
print({'device': DEVICE, 'parameters': params, 'embedding_dim': EMBED_DIM, 'embedding_bytes_fp32': embedding_bytes, 'approx_flops': flops, 'batch1_latency_ms': round(latency_ms,2)})